In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import holidays

In [2]:
# Cargar los datos de ventas y competencia desde data/raw/entrenamiento

# Cargar los archivos CSV en DataFrames
ventas_df = pd.read_csv("../data/raw/entrenamiento/ventas.csv")
competencia_df = pd.read_csv("../data/raw/entrenamiento/competencia.csv")

# Mostrar las primeras filas para verificar
print("Ventas:")
print(ventas_df.head())
print("\nCompetencia:")
print(competencia_df.head())

Ventas:
        fecha producto_id                            nombre categoria  \
0  2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1  2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2  2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3  2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4  2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True                  6   
1  Zapatillas Running          135         True                 10   
2  Zapatillas Running           85        False                  2   
3  Zapatillas Running           75        False                  2   
4        Ropa Running           35        False                  2   

   precio_venta  ingresos  
0        118.36    710.16  
1        136.82   1368.20  
2         84.93    169.86  
3         75.42    1

In [6]:
resumen_calidad = pd.DataFrame({
    "tipo": ventas_df.dtypes.astype(str),
    "no_nulos": ventas_df.notna().sum(),
    "nulos": ventas_df.isna().sum(),
    "porcentaje_nulos": (ventas_df.isna().mean() * 100).round(2),
    "valores_unicos": ventas_df.nunique(dropna=True),
})

filas_duplicadas = ventas_df.duplicated().sum()
duplicados_fecha_producto = ventas_df.duplicated(
    subset=["fecha", "producto_id"]
).sum()
fechas_parseadas = pd.to_datetime(ventas_df["fecha"], errors="coerce")
fechas_invalidas = fechas_parseadas.isna().sum()
columnas_texto = [
    columna
    for columna in ventas_df.columns
    if pd.api.types.is_string_dtype(ventas_df[columna].dtype)
]
cadenas_vacias = ventas_df[columnas_texto].apply(
    lambda columna: columna.str.strip().eq("").sum()
)
columnas_numericas = ventas_df.select_dtypes(include="number").columns
valores_negativos = ventas_df[columnas_numericas].lt(0).sum()
inconsistencias_ingresos = (
    ventas_df["ingresos"]
    - ventas_df["precio_venta"] * ventas_df["unidades_vendidas"]
).abs().gt(0.01).sum()
descriptivos = ventas_df.describe(include="all").T

print(f"Dimensiones: {ventas_df.shape[0]} filas x {ventas_df.shape[1]} columnas")
print(f"Rango de fechas: {fechas_parseadas.min().date()} a {fechas_parseadas.max().date()}")
print(f"Filas duplicadas completas: {filas_duplicadas}")
print(f"Claves fecha-producto duplicadas: {duplicados_fecha_producto}")
print(f"Fechas no interpretables: {fechas_invalidas}")
print(f"Inconsistencias ingresos vs. precio_venta x unidades: {inconsistencias_ingresos}")
print("\nCadenas vacías por columna de texto:")
print(cadenas_vacias.to_string())
print("\nValores negativos por columna numérica:")
print(valores_negativos.to_string())
print("\nCalidad por variable:")
print(resumen_calidad.to_string())
print("\nEstadísticos descriptivos:")
print(descriptivos.to_string())

Dimensiones: 3552 filas x 10 columnas
Rango de fechas: 2021-10-25 a 2024-11-30
Filas duplicadas completas: 0
Claves fecha-producto duplicadas: 0
Fechas no interpretables: 0
Inconsistencias ingresos vs. precio_venta x unidades: 0

Cadenas vacías por columna de texto:
fecha           0
producto_id     0
nombre          0
categoria       0
subcategoria    0

Valores negativos por columna numérica:
precio_base          0
unidades_vendidas    0
precio_venta         0
ingresos             0

Calidad por variable:
                      tipo  no_nulos  nulos  porcentaje_nulos  valores_unicos
fecha                  str      3552      0               0.0             148
producto_id            str      3552      0               0.0              24
nombre                 str      3552      0               0.0              24
categoria              str      3552      0               0.0               4
subcategoria           str      3552      0               0.0              16
precio_base        